In [0]:
-- =============================================================================
-- TRAVEL BOOKING SCD2 MERGE PROJECT - DAILY REVENUE ANALYTICS
-- =============================================================================
-- This script creates and populates daily revenue analytics by booking type
-- Purpose: Provides daily revenue metrics for business reporting and analysis
-- Business Value: Revenue tracking, booking type performance, daily trends
-- Dependencies: Requires bronze.booking_inc table to be populated

-- =============================================================================
-- DAILY REVENUE TABLE CREATION
-- =============================================================================
-- Create table for daily revenue analytics with auto-optimization
-- business_date: Date dimension for time-based analysis
-- booking_type: Booking type dimension for segmentation
-- total_amount: Net revenue after discounts
-- total_quantity: Total number of bookings

create table if not exists travel_bookings.analytics.daily_revenue_by_type
(
business_date date, booking_type string, total_amount double, total_quantity bigint
) using delta 
TBLPROPERTIES (
  delta.autoOptimize.optimizeWrite=true, 
  delta.autoOptimize.autoCompact=true
  );

-- =============================================================================
-- IDEMPOTENT DATA LOADING
-- =============================================================================
-- Delete existing data for the specified date to ensure idempotency
-- Uses parameter binding for flexible date specification
-- Falls back to current date if no parameter provided

delete from travel_bookings.analytics.daily_revenue_by_type where business_date=coalesce(cast(arrival_date as date), current_date());

-- =============================================================================
-- DAILY REVENUE CALCULATION
-- =============================================================================
-- Calculate daily revenue metrics from bronze booking data
-- Net amount: amount - discount for accurate revenue calculation
-- Aggregates by booking type for business segmentation
-- Uses explicit casting for data type consistency

insert into travel_bookings.analytics.daily_revenue_by_type 
select  coalesce(try_cast(:arrival_date as date), current_date()) business_date, booking_type, sum(amount)-sum(discount) as total_amount, sum(quantity) total_quantity 
from travel_bookings.bronze.booking_inc where business_date=coalesce(arrival_date, current_date())
group by booking_type;